# Exploration du seq2seq

Le code vit dans le paquet `fatia/` : ce notebook sert à **inspecter et visualiser**, pas à
héberger la logique. L'entraînement se lance depuis le terminal :

```bash
python entrainer.py --intentions data/intentions.json --sortie modeles --epochs 100
```

In [ ]:
import sys, json
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

from fatia import donnees

print("Racine du projet :", RACINE)

## 1. Les données

Le split se fait **au niveau des exemples, avant le produit croisé** : sinon une même question
se retrouverait des deux côtés (une fois par réponse de son intention) et la loss de validation
ne détecterait plus le sur-apprentissage.

In [ ]:
preparees = donnees.preparer(RACINE / "data" / "intentions.json")
vocabulaire = preparees["vocabulaire"]

print(f"Vocabulaire       : {len(vocabulaire)} tokens")
print(f"Paires train      : {len(preparees['train'])}")
print(f"Paires validation : {len(preparees['validation'])}")

questions_train = {q for q, _ in preparees["paires_train"]}
questions_val = {q for q, _ in preparees["paires_validation"]}
print(f"\nQuestions communes train/validation : {len(questions_train & questions_val)} (doit être 0)")

In [ ]:
for entree, cible in preparees["train"][:3]:
    print("entrée :", vocabulaire.decoder(entree))
    print("cible  :", vocabulaire.decoder(cible))
    print()

## 2. Courbes d'apprentissage

À lire attentivement : la loss de validation doit finir par **remonter** pendant que celle
d'entraînement continue de descendre. Si les deux restent collées, c'est le signe que la fuite
de données n'a pas été évitée — et le checkpoint sélectionné ne voudrait alors rien dire.

In [ ]:
import matplotlib.pyplot as plt

historique = json.loads((RACINE / "modeles" / "historique.json").read_text(encoding="utf-8"))
epochs = [h["epoch"] for h in historique]
train = [h["perte_train"] for h in historique]
validation = [h["perte_validation"] for h in historique]

meilleur = min(historique, key=lambda h: h["perte_validation"])

plt.figure(figsize=(9, 5))
plt.plot(epochs, train, label="train")
plt.plot(epochs, validation, label="validation")
plt.axvline(meilleur["epoch"], color="red", linestyle="--",
            label=f"checkpoint conservé (epoch {meilleur['epoch']})")
plt.xlabel("epoch"); plt.ylabel("entropie croisée"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Apprentissage du seq2seq")
plt.show()

print(f"Meilleure validation : {meilleur['perte_validation']:.4f} à l'epoch {meilleur['epoch']}")

## 3. Génération

Rappel : comme chaque question a plusieurs réponses cibles, le modèle converge vers la *moyenne*
de ces réponses. Attends-toi à des hybrides bancals plutôt qu'à un tirage propre entre les
29 réponses — c'est le comportement normal de cette architecture sur ce dataset, pas un bug.

In [ ]:
from fatia.generation import Repondeur

repondeur = Repondeur(RACINE / "modeles")
print(f"Checkpoint : epoch {repondeur.epoch}, val {repondeur.perte_validation:.4f}\n")

phrases = [
    "salut",                      # connue
    "comment tu vas",             # connue
    "merci beaucoup",             # connue
    "bonjour mon cher ami",       # inédite, mots connus
    "xyzzy plugh frobnicate",     # entièrement hors-vocabulaire -> repli
]

for phrase in phrases:
    glouton = repondeur.repondre(phrase, temperature=0.0)
    echantillon = repondeur.repondre(phrase, temperature=0.8)
    print(f"» {phrase}")
    print(f"   glouton      : {glouton['reponse']}")
    print(f"   température  : {echantillon['reponse']}")
    print(f"   inconnus     : {glouton['tokens_inconnus']}/{glouton['tokens_total']}"
          f"{'  (repli déclenché)' if glouton['repli'] else ''}\n")

## 4. Test de sur-apprentissage

Le test le plus utile du projet. Sur 20 paires question→réponse **cohérentes**, le modèle doit
restituer les réponses mot pour mot. S'il y arrive, la chaîne (tokenizer, encodeur, décodeur,
teacher forcing, décodage) est correcte et ce qui manque, ce sont les données. S'il échoue,
le bug est dans le code.

```bash
python entrainer.py --paires data/paires_test.json \
    --sortie modeles/test_surapprentissage --epochs 400
```

La sortie est isolée : elle n'écrase jamais `modeles/seq2seq.pt` ni son vocabulaire.

In [ ]:
paires_test = json.loads((RACINE / "data" / "paires_test.json").read_text(encoding="utf-8"))
repondeur_test = Repondeur(RACINE / "modeles" / "test_surapprentissage")

exacts = 0
for question, attendue in paires_test:
    obtenue = repondeur_test.repondre(question, temperature=0.0)["reponse"]
    identique = obtenue.lower().replace(" ", "") == attendue.lower().replace(" ", "")
    exacts += identique
    print(f"{'OK ' if identique else 'NON'} » {question}")
    if not identique:
        print(f"      attendu : {attendue}")
        print(f"      obtenu  : {obtenue}")

print(f"\n{exacts}/{len(paires_test)} réponses mémorisées exactement")